# 3 · Transfer to Vava'u

The three Tongatapu models (pixel, superpixel and embedding Random Forests) are applied, unchanged, to **Vava'u**, about 300 km north. For a classifier to transfer, its inputs must be made the same way. So the Vava'u composite repeats every step of notebook 1, and the depth-invariant indices use the **Tongatapu** attenuation ratios, as `IMAGE_TRAIN` does in GEE script 2.

There are no drop-camera data for Vava'u, so we can't measure accuracy here. Instead we ask **where the Tongatapu model applies**: the area of applicability, computed against the Tongatapu training drops, shows which Vava'u pixels resemble what the model was trained on. Then we compare class areas with Tongatapu, inside and outside it.

**Inputs:** `data/rf_models.joblib`, `data/tongatapu_classes_*.tif` (notebook 2)
**Outputs:** `data/vavau_composite.nc`, `data/vavau_classes_*.tif`, `data/vavau_aoa_*.tif` (one per method)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # noqa: F401
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import odc.stac
from pystac_client import Client
from dask.diagnostics import ProgressBar
from scipy import ndimage
from skimage.feature import canny
from skimage.filters import threshold_otsu
from skimage.color import rgb2hsv
from scipy.spatial import cKDTree
from skimage.segmentation import slic
import geopandas as gpd
from shapely.geometry import box
from rasterio.enums import Resampling
import urllib.request
import joblib

AOI = (-174.03, -18.78, -173.91, -18.60)   # lon/lat: west, south, east, north
START, END = "2023-05-01", "2023-10-31"   # dry season
MAX_CLOUD = 30
CRS = "EPSG:32701"
RES = 10
STAC_URL = "https://earth-search.aws.element84.com/v1"
COLLECTION = "sentinel-2-c1-l2a"
BANDS = {"B1": "coastal", "B2": "blue", "B3": "green", "B4": "red", "B5": "rededge1", "B8A": "nir08", "B11": "swir16"}

DATA = Path("data")
m = joblib.load(DATA / "rf_models.joblib")
models, aoa_parts, FEATURES, CLASSES, COLOURS, k_ratio = m["models"], m["aoa"], m["features"], m["classes"], m["colours"], m["k_ratio"]
seg_settings, aef = m["segmentation"], m["aef"]
METHODS = list(models)
TAGS = {"Pixel RF": "pixel", "Object RF": "object", "Embedding RF": "embedding"}

## 1. Composite: same steps as notebook 1

Same dates, cloud filter, cloud mask, 25th percentile, edge-based Otsu threshold for land (fitted again for Vava'u), and the deep water index with a fixed threshold. The deep water threshold depends on the scene, so check the histogram and adjust `DEEP_THRESHOLD` if needed.

In [ ]:
items = Client.open(STAC_URL).search(collections=[COLLECTION], bbox=AOI, datetime=f"{START}/{END}",
                                     query={"eo:cloud_cover": {"lte": MAX_CLOUD}}).item_collection()
ds = odc.stac.load(items, bands=list(BANDS.values()) + ["scl"], bbox=AOI, crs=CRS, resolution=RES,
                   groupby="solar_day", chunks={"x": 1024, "y": 1024})
ds = ds.rename({v: k for k, v in BANDS.items()} | {"scl": "SCL"})
print(f"{len(items)} items on {ds.sizes['time']} dates")

window = 2 * round(50 / RES) + 1
cloud = ds["SCL"].isin([0, 1, 3, 8, 9, 10]).astype("uint8").rolling(x=window, y=window, center=True, min_periods=1).max() == 1
bands = ds[list(BANDS)]
refl = (bands.where(bands > 0) / 10000 - 0.1).where(~cloud)
with ProgressBar():
    comp = refl.chunk(time=-1).quantile(0.25, dim="time", skipna=True).drop_vars("quantile").compute()
comp = comp.astype("float32").clip(min=0)    # as in notebook 1: no negative reflectance

In [ ]:
def edge_otsu(index, initial_threshold, min_edge_px, buffer_px):
    """Otsu threshold of the index, using only pixels near long edges of an initial split."""
    values = index.values
    ok = np.isfinite(values)                                   # NaN areas (masked, no data) are ignored
    split = np.where(ok, values, initial_threshold) < initial_threshold
    edges = canny(split.astype(float), sigma=1, mask=ok)
    labels, n = ndimage.label(edges, structure=np.ones((3, 3)))
    length = ndimage.sum(edges, labels, index=np.arange(1, n + 1))
    long_edges = np.isin(labels, np.flatnonzero(length >= min_edge_px) + 1)
    near = ndimage.distance_transform_edt(~long_edges) <= buffer_px
    return threshold_otsu(values[near & ok]), near

dark = np.minimum(comp["B11"], comp["B8A"])
mndwi = (comp["B3"] - dark) / (comp["B3"] + dark)
land_threshold, _ = edge_otsu(mndwi, 0, 80, round(90 / RES))
water = (mndwi >= land_threshold) & comp["B2"].notnull()

DEEP_THRESHOLD = 0.2        # deep water index, as notebook 1; tune for this composite
hsv = rgb2hsv(np.clip(np.nan_to_num(np.dstack([comp["B1"], comp["B2"], comp["B3"]])), 0, 1))
dwi = comp["B3"].copy(data=(hsv[..., 1] - hsv[..., 0]) / (hsv[..., 1] + hsv[..., 0])).where(water)
deep = (dwi > DEEP_THRESHOLD) & water
shallow = water & ~deep
print(f"land/water threshold: {land_threshold:.3f}; deep threshold {DEEP_THRESHOLD}: "
      f"{float(deep.sum() / water.sum()):.0%} of water masked as optically deep")

out = comp.assign(water=water.astype("uint8"), shallow=shallow.astype("uint8")).rio.write_crs(CRS)
(DATA / "vavau_composite.nc").unlink(missing_ok=True)
out.to_netcdf(DATA / "vavau_composite.nc", engine="h5netcdf",
              encoding={b: {"dtype": "int16", "scale_factor": 1e-4, "_FillValue": -32768, "zlib": True} for b in BANDS})

def rgb(img, gain):
    return np.clip(np.nan_to_num(np.dstack([img["B4"], img["B3"], img["B2"]]) * gain), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [0.7, 1]})
v = dwi.values[np.isfinite(dwi.values)]
axes[0].hist(v, bins=200, color="0.6")
axes[0].axvline(DEEP_THRESHOLD, color="r", label=f"DEEP_THRESHOLD = {DEEP_THRESHOLD}")
axes[0].set(xlabel="deep water index, water pixels", yticks=[]); axes[0].legend()
axes[1].imshow(rgb(comp.where(water), 4))
axes[1].imshow(np.where(deep, 1.0, np.nan), cmap=ListedColormap(["#08306b"]), alpha=0.7)
axes[1].set_title("Vava'u: land masked, optically deep in dark blue"); axes[1].axis("off")
plt.tight_layout()

## 2. Classify with the three Tongatapu models

Each model gets its inputs made exactly as in notebook 2:
* **Pixel RF:** the spectral features, with the Tongatapu Lyzenga ratios
* **Object RF:** superpixels of the Vava'u composite (same settings), then the same features averaged per superpixel
* **Embedding RF:** the 2023 AlphaEarth embeddings for Vava'u. The new AOI straddles the boundary between UTM zones 1 and 2 at 174° W, so tiles from both zones are read, and each is resampled onto our grid.

In [ ]:
def norm_diff(a, b):
    return (a - b) / (a + b)

log_r = np.log(comp[["B1", "B2", "B3", "B4"]].where(comp[["B1", "B2", "B3", "B4"]] > 0))
feats = comp[["B1", "B2", "B3", "B4", "B5"]].copy()
for (bi, bj), k in k_ratio.items():
    feats[f"DII{bi[1]}{bj[1]}"] = (log_r[bi] - k * log_r[bj]) / np.sqrt(k ** 2 + 1)
feats["NDVI"] = norm_diff(comp["B8A"], comp["B4"])
feats["ReNDVI"] = norm_diff(comp["B5"], comp["B4"])
feats["RedGreen"] = norm_diff(comp["B4"], comp["B3"])

table = feats[FEATURES].to_dataframe()[FEATURES].astype("float32")
ok = shallow.values.ravel() & np.isfinite(table.to_numpy()).all(axis=1)
ny, nx = comp.sizes["y"], comp.sizes["x"]

shallow_grid = ok.reshape(ny, nx)
img = np.dstack([comp[b].values for b in seg_settings["bands"]]).astype("float32")
img = (img - img[shallow_grid].mean(axis=0)) / img[shallow_grid].std(axis=0)
img[~shallow_grid] = -10
seg = slic(img, n_segments=ny * nx // seg_settings["segment_px"], compactness=seg_settings["compactness"],
           channel_axis=-1, start_label=1, max_num_iter=5).ravel()
seg_table = table[ok].groupby(seg[ok]).mean()
print(f"{ok.sum():,} shallow pixels, {len(seg_table):,} superpixels")

In [ ]:
AEF_ROOT, AEF_YEAR = aef["root"], aef["year"]
if not (DATA / "aef_index.parquet").exists():
    urllib.request.urlretrieve(f"{AEF_ROOT}/aef_index.parquet", DATA / "aef_index.parquet")
aef_index = gpd.read_parquet(DATA / "aef_index.parquet")

def load_embeddings(aoi, like):
    """AlphaEarth embeddings (64 bands, int8, -128 = no data) on the grid of `like`."""
    tiles = aef_index[(aef_index["year"] == AEF_YEAR) & aef_index.intersects(box(*aoi))]
    path = next(c for c in tiles.columns if tiles[c].astype(str).str.endswith(".tiff").all())
    out = None
    for p in tiles[path]:
        with rioxarray.open_rasterio(f"{AEF_ROOT}/{p.split('annual/')[-1]}", chunks=True) as src:
            part = src.rio.clip_box(*aoi, crs="EPSG:4326").load().rio.write_nodata(-128)
        part = part.rio.reproject_match(like, resampling=Resampling.nearest)
        out = part if out is None else out.where(out != -128, part)
    print(f"{len(tiles)} embedding tile(s) read")
    return out

def dequantise(q):
    q = q.astype("float32")
    return (q / 127.5) ** 2 * np.sign(q)

E = load_embeddings(AOI, comp["B2"].rio.write_crs(CRS)).values.reshape(64, -1)
has_emb = (E != -128).all(axis=0)
print(f"embeddings available for {has_emb[ok].mean():.0%} of shallow pixels")
EMB = [f"A{i:02d}" for i in range(64)]

In [ ]:
def dissimilarity(parts, X):
    tree, X = cKDTree(parts["train_scaled"]), np.asarray(X, dtype="float32")
    di = [tree.query(((X[i:i + 200_000] - parts["mu"]) / parts["sd"] * parts["weights"])[:, parts["keep"]])[0]
          for i in range(0, len(X), 200_000)]
    return np.concatenate(di) / parts["d_mean"]

def to_grid(values, where, fill, dtype):
    full = np.full(ok.size, fill, dtype=dtype)
    full[where] = values
    return xr.DataArray(full.reshape(ny, nx), coords={"y": comp.y, "x": comp.x})

code = {c: i for i, c in enumerate(CLASSES)}
maps, dis = {}, {}
maps["Pixel RF"] = to_grid(pd.Series(models["Pixel RF"].predict(table[ok])).map(code).values, ok, 255, "uint8")
dis["Pixel RF"] = to_grid(dissimilarity(aoa_parts["Pixel RF"], table[ok]), ok, np.nan, "float32")

seg_class = pd.Series(models["Object RF"].predict(seg_table), index=seg_table.index).map(code)
seg_di = pd.Series(dissimilarity(aoa_parts["Object RF"], seg_table), index=seg_table.index)
maps["Object RF"] = to_grid(seg_class.reindex(seg[ok]).values, ok, 255, "uint8")
dis["Object RF"] = to_grid(seg_di.reindex(seg[ok]).values, ok, np.nan, "float32")

ok_emb = ok & has_emb
E_ok = pd.DataFrame(dequantise(E[:, ok_emb].T), columns=EMB)
maps["Embedding RF"] = to_grid(pd.Series(models["Embedding RF"].predict(E_ok)).map(code).values, ok_emb, 255, "uint8")
dis["Embedding RF"] = to_grid(dissimilarity(aoa_parts["Embedding RF"], E_ok), ok_emb, np.nan, "float32")
del E_ok

for name, tag in TAGS.items():
    (DATA / f"vavau_classes_{tag}.tif").unlink(missing_ok=True)
    maps[name].rio.write_crs(CRS).rio.write_nodata(255).rio.to_raster(DATA / f"vavau_classes_{tag}.tif")
    (DATA / f"vavau_aoa_{tag}.tif").unlink(missing_ok=True)
    dis[name].rio.write_crs(CRS).rio.write_nodata(np.nan).rio.to_raster(DATA / f"vavau_aoa_{tag}.tif")
print("saved data/vavau_classes_*.tif and data/vavau_aoa_*.tif (dissimilarity index; inside the AOA where DI <= threshold)")

## 3. Area of applicability at Vava'u

Each pixel's dissimilarity to the **Tongatapu** training drops, in each method's own feature space, using the scaling, weights and thresholds saved in notebook 2. Nothing is refitted: the question is how similar Vava'u is to what each model learned from. The methods can differ a lot here. A method whose features are more alike between the two islands transfers more of its map.

In [ ]:
kw = dict(cmap=ListedColormap(COLOURS), vmin=-0.5, vmax=len(CLASSES) - 0.5, interpolation="nearest")
inside = {name: (dis[name] <= aoa_parts[name]["threshold"]).values for name in METHODS}
fig, axes = plt.subplots(1, 3, figsize=(22, 10))
for ax, name in zip(axes, METHODS):
    m = maps[name]
    ax.imshow(rgb(comp, 4))
    ax.imshow(np.where(inside[name], m, np.nan), **kw)
    ax.imshow(np.where(dis[name].notnull() & ~inside[name], 1.0, np.nan), cmap=ListedColormap(["#ff00ff"]), alpha=0.8, interpolation="nearest")
    share = inside[name][dis[name].notnull().values].mean()
    ax.set_title(f"{name}: {share:.0%} inside AOA (magenta = outside)"); ax.axis("off")
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in COLOURS], labels=CLASSES, loc="lower left", fontsize=8)
plt.tight_layout()

For the Pixel RF, the features have physical meaning, so we can ask *why* Vava'u pixels fall outside: compare their distributions with the Tongatapu training drops. Deeper or clearer water, different brightness, or substrates not sampled at Tongatapu all show up here.

In [ ]:
parts = aoa_parts["Pixel RF"]
v_in = inside["Pixel RF"].ravel()[ok]
top = pd.Series(parts["weights"], FEATURES).nlargest(4).index
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, f in zip(axes, top):
    v = table.loc[ok, f].to_numpy()
    bins = np.linspace(*np.percentile(np.r_[v, parts["train_features"][f]], [1, 99]), 50)
    ax.hist(v[v_in], bins=bins, density=True, alpha=0.5, label="Vava'u inside")
    ax.hist(v[~v_in], bins=bins, density=True, alpha=0.5, color="m", label="Vava'u outside")
    ax.hist(parts["train_features"][f], bins=bins, density=True, histtype="step", color="k", lw=2, label="Tongatapu training")
    ax.set(title=f, yticks=[])
axes[0].legend(fontsize=8)
plt.tight_layout()

## 4. Compare the methods, and with Tongatapu

Different class proportions may be real, since Vava'u is a different reef system. They may also mean a model is meeting conditions it was never trained on. Restricting the comparison to pixels inside each method's area of applicability separates the two. Classes that only appear, or grow, *outside* it are the least trustworthy.

In [ ]:
def shares(values):
    v = values[values != 255]
    return pd.Series(v).map(dict(enumerate(CLASSES))).value_counts(normalize=True).reindex(CLASSES, fill_value=0)

table_shares = {}
for name, tag in TAGS.items():
    with rioxarray.open_rasterio(DATA / f"tongatapu_classes_{tag}.tif") as r:
        table_shares[(name, "Tongatapu")] = shares(r.values.ravel())
    m = maps[name].values
    table_shares[(name, "Vava'u inside AOA")] = shares(np.where(inside[name], m, 255).ravel())
    table_shares[(name, "Vava'u outside AOA")] = shares(np.where(dis[name].notnull().values & ~inside[name], m, 255).ravel())
table_shares = pd.DataFrame(table_shares)

ax = table_shares.T.iloc[::-1].plot.barh(stacked=True, color=COLOURS, figsize=(11, 6), edgecolor="k", lw=0.3)
ax.set_yticklabels([f"{a}: {b}" for a, b in table_shares.columns[::-1]])
ax.set(xlabel="share of classified area"); ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
(table_shares * 100).round(1)

In [ ]:
stack = np.stack([maps[name].values for name in METHODS])
valid = (stack != 255).all(axis=0)
n_classes = 1 + (stack[0] != stack[1]) + ((stack[2] != stack[0]) & (stack[2] != stack[1]))
all_inside = np.logical_and.reduce([inside[name] for name in METHODS]) & valid

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(rgb(comp, 4))
ax.imshow(np.where(valid, n_classes, np.nan), cmap=ListedColormap(["#1a9850", "#fee08b", "#d73027"]), vmin=0.5, vmax=3.5, interpolation="nearest")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#1a9850", "#fee08b", "#d73027"]],
          labels=["all three agree", "two agree", "all differ"], loc="lower left")
ax.set_title("Agreement between the three Vava'u maps"); ax.axis("off")
print(f"all three agree on {(n_classes[valid] == 1).mean():.0%} of pixels; "
      f"on {(n_classes[all_inside] == 1).mean():.0%} of pixels inside all three areas of applicability")

## Exercises
1. **Which map would you hand to a manager?** Weigh test accuracy at Tongatapu (notebook 2), the share of Vava'u inside each area of applicability, and agreement between methods.
2. **Where to survey.** Pixels outside all three areas of applicability, or where the methods disagree, are where a first drop-camera survey at Vava'u would help most. Find them.
3. **Deep water threshold.** Change `DEEP_THRESHOLD`. Do the pixels it adds or removes fall inside or outside the areas of applicability?
4. **Is it the water?** If depth-sensitive features (B1–B3) dominate the differences in section 3, would retraining the Pixel RF on the DIIs alone transfer better? Try it in notebook 2.